In [1]:
pip install opencv-python numpy scikit-learn torch torchvision scikit-image pillow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import cv2
import numpy as np
from collections import defaultdict
import os
from sklearn.cluster import DBSCAN
import torch
import torchvision.transforms as T
from PIL import Image
from skimage.measure import label, regionprops

# ---------- 工具函数 ----------
def classify_hue(h_value):
    """Classify H value into simplified color categories"""
    hue_degrees = h_value * 2  # 0-179 -> 0-360°
    if hue_degrees < 15 or hue_degrees >= 345:
        return "Red"
    elif 15 <= hue_degrees < 45:
        return "Orange/Yellow"
    elif 45 <= hue_degrees < 165:
        return "Green"
    elif 165 <= hue_degrees < 255:
        return "Blue"
    else:  # 255-345
        return "Purple"

def classify_saturation(s_percent):
    """Classify based on saturation percentage"""
    if s_percent < 0.3:
        return "Low Saturation", s_percent * 100
    elif 0.3 <= s_percent < 0.7:
        return "Medium Saturation", s_percent * 100
    else:
        return "High Saturation", s_percent * 100

def classify_value(v_percent):
    """Classify based on value (brightness) percentage"""
    if v_percent < 0.3:
        return "Low Brightness", v_percent * 100
    elif 0.3 <= v_percent < 0.7:
        return "Medium Brightness", v_percent * 100
    else:
        return "High Brightness", v_percent * 100

def get_color_bgr(color_name):
    """Return BGR value for drawing based on color name"""
    color_map = {
        "Red": (0, 0, 255),
        "Orange/Yellow": (0, 165, 255),
        "Green": (0, 255, 0),
        "Blue": (255, 0, 0),
        "Purple": (255, 0, 255),
        "Other": (192, 192, 192)
    }
    return color_map.get(color_name, (192, 192, 192))

def hue_difference(h1, h2):
    """环形色调差值（角度制）"""
    diff = abs(h1 - h2)
    return min(diff, 180 - diff)

# ---------- A: 传统计算机识别 ----------
def traditional_recognition(image, min_area=100, morph_kernel_size=5, merge_small=True):
    """
    基于HSV颜色空间的传统色块检测。
    返回：all_blocks列表，每个元素为字典，包含color, hue, saturation, brightness,
          area, position (x,y,w,h), contour, mean_hsv, type等。
    """
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    height, width = image.shape[:2]
    result_image = image.copy()

    # 存储所有轮廓及其属性（保留轮廓以便后续生成掩码）
    all_contours_data = []  # (contour, mean_hsv, area, center, original_color_name)

    color_ranges = [
        ("Red", [(0, 50, 50), (10, 255, 255)], [(170, 50, 50), (179, 255, 255)]),
        ("Orange/Yellow", [(11, 50, 50), (37, 255, 255)]),
        ("Green", [(38, 50, 50), (82, 255, 255)]),
        ("Blue", [(83, 50, 50), (127, 255, 255)]),
        ("Purple", [(128, 50, 50), (169, 255, 255)])
    ]

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_kernel_size, morph_kernel_size))

    for color_name, *ranges in color_ranges:
        mask = np.zeros((height, width), dtype=np.uint8)
        for range_list in ranges:
            if len(range_list) == 2:
                lower, upper = range_list
                color_mask = cv2.inRange(hsv, np.array(lower), np.array(upper))
                mask = cv2.bitwise_or(mask, color_mask)

        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for contour in contours:
            area = cv2.contourArea(contour)
            contour_mask = np.zeros((height, width), dtype=np.uint8)
            cv2.drawContours(contour_mask, [contour], -1, 255, -1)
            mean_hsv = cv2.mean(hsv, mask=contour_mask)[:3]

            M = cv2.moments(contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
            else:
                cx, cy = 0, 0

            all_contours_data.append((contour, mean_hsv, area, (cx, cy), color_name))

    # 分离大块和小块
    large_blocks = [d for d in all_contours_data if d[2] >= min_area]
    small_blocks = [d for d in all_contours_data if d[2] < min_area]

    merged_blocks = []  # 存放合并后的块

    if merge_small and len(small_blocks) > 0:
        print(f"Merging {len(small_blocks)} small blocks...")

        # 构建特征矩阵
        features = []
        for data in small_blocks:
            contour, mean_hsv, area, center, orig_color = data
            h, s, v = mean_hsv
            h_deg = h * 2
            h_norm = h_deg / 360.0
            s_norm = s / 255.0
            v_norm = v / 255.0
            x_norm = center[0] / width
            y_norm = center[1] / height
            features.append([h_norm, s_norm, v_norm, x_norm, y_norm])

        features = np.array(features)

        # DBSCAN 聚类
        color_thresh = 15   # 颜色差异阈值（角度）
        dist_thresh = 50    # 空间距离阈值（像素）
        eps_color = color_thresh / 360.0
        eps_dist = dist_thresh / max(height, width)
        eps = np.sqrt(eps_color**2 * 3 + eps_dist**2 * 2)
        clustering = DBSCAN(eps=eps, min_samples=1).fit(features)
        labels = clustering.labels_

        # 标记哪些小块已被成功合并（即属于某个簇且合并后面积≥min_area）
        used_small = np.zeros(len(small_blocks), dtype=bool)

        # 处理每个簇（label != -1）
        unique_labels = set(labels)
        for label in unique_labels:
            if label == -1:
                continue
            indices = np.where(labels == label)[0]
            # 合并该簇所有轮廓
            mask = np.zeros((height, width), dtype=np.uint8)
            for idx in indices:
                contour = small_blocks[idx][0]
                cv2.drawContours(mask, [contour], -1, 255, -1)
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if len(contours) == 0:
                continue
            merged_contour = contours[0]  # 通常只有一个
            area = cv2.contourArea(merged_contour)
            if area >= min_area:
                # 计算平均HSV
                contour_mask = np.zeros((height, width), dtype=np.uint8)
                cv2.drawContours(contour_mask, [merged_contour], -1, 255, -1)
                mean_hsv = cv2.mean(hsv, mask=contour_mask)[:3]
                # 中心点
                M = cv2.moments(merged_contour)
                cx = int(M["m10"] / M["m00"]) if M["m00"] != 0 else 0
                cy = int(M["m01"] / M["m00"]) if M["m00"] != 0 else 0
                merged_blocks.append((merged_contour, mean_hsv, area, (cx, cy), "Merged"))
                used_small[indices] = True

        # 处理未被合并的小块（剩余小块）
        remaining_indices = np.where(~used_small)[0]
        if len(remaining_indices) > 0:
            print(f"Combining {len(remaining_indices)} remaining small blocks into one block using convex hull...")
            # 收集所有剩余小块的轮廓点
            all_points = []
            for idx in remaining_indices:
                contour = small_blocks[idx][0]
                all_points.extend(contour.reshape(-1, 2).tolist())
            if len(all_points) >= 3:
                points = np.array(all_points, dtype=np.int32)
                hull = cv2.convexHull(points)
                area = cv2.contourArea(hull)
                # 计算平均HSV（基于所有剩余小块的并集掩膜）
                mask = np.zeros((height, width), dtype=np.uint8)
                for idx in remaining_indices:
                    contour = small_blocks[idx][0]
                    cv2.drawContours(mask, [contour], -1, 255, -1)
                mean_hsv = cv2.mean(hsv, mask=mask)[:3]
                # 中心点（使用凸包的中心）
                M = cv2.moments(hull)
                cx = int(M["m10"] / M["m00"]) if M["m00"] != 0 else 0
                cy = int(M["m01"] / M["m00"]) if M["m00"] != 0 else 0
                merged_blocks.append((hull, mean_hsv, area, (cx, cy), "Remaining"))
            else:
                print("Warning: Too few remaining points to form convex hull, skipping.")

        print(f"Generated {len(merged_blocks)} merged blocks (including remaining).")

    # 合并所有有效块
    all_valid = large_blocks + merged_blocks

    # 构建最终块信息列表，包含轮廓以便后续计算重叠
    all_blocks = []
    for data in all_valid:
        contour, mean_hsv, area, center, src_type = data
        h_value, s_value, v_value = mean_hsv
        s_percent = s_value / 255.0
        v_percent = v_value / 255.0
        detected_color = classify_hue(h_value)
        sat_level, sat_percent = classify_saturation(s_percent)
        val_level, val_percent = classify_value(v_percent)
        x, y, w, h = cv2.boundingRect(contour)

        block_info = {
            "color": detected_color,
            "type": src_type,          # 原始颜色名 / "Merged" / "Remaining"
            "hue": h_value,
            "saturation": (sat_level, sat_percent),
            "brightness": (val_level, val_percent),
            "area": area,
            "position": (x, y, w, h),
            "contour": contour,         # 轮廓，用于生成掩码
            "mean_hsv": mean_hsv         # 保存均值方便后续使用
        }
        all_blocks.append(block_info)

    # 输出统计信息
    print("\n===== 传统识别完成 =====")
    print(f"有效色块数量: {len(all_blocks)}")
    return all_blocks, hsv, image

# ---------- B: AI识别 ----------
def ai_recognition(image):
    """
    使用 DeepLabV3-ResNet50 对图像进行语义分割。
    返回：ai_regions列表，每个元素为字典：
          - class_id: 类别ID (0背景, 1~20为COCO类别)
          - class_name: 类别名称
          - mask: 二值掩码 (H,W) bool
          - contour: 该区域的轮廓
          - hsv_mean: 该区域的平均HSV
          - area: 面积
    """
    # 加载模型（首次运行会自动下载权重）
    model = torch.hub.load('pytorch/vision', 'deeplabv3_resnet50', pretrained=True)
    model.eval()

    # 图像预处理
    preprocess = T.Compose([
        T.ToPILImage(),
        T.Resize(520),                     # 模型要求的输入尺寸
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # 将OpenCV BGR图像转为RGB
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    input_tensor = preprocess(rgb).unsqueeze(0)  # 增加batch维度

    with torch.no_grad():
        output = model(input_tensor)['out'][0]   # shape (21, H, W)
    prediction = output.argmax(0).byte().cpu().numpy()  # (H, W)

    # 将预测结果缩放到原图尺寸
    pred_resized = cv2.resize(prediction, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)

    # COCO类别名称（21类，索引0为背景）
    coco_classes = [
        '__background__', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
        'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant',
        'sheep', 'sofa', 'train', 'tvmonitor'
    ]

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    height, width = image.shape[:2]
    ai_regions = []

    unique_classes = np.unique(pred_resized)
    for cls in unique_classes:
        if cls == 0:   # 跳过背景
            continue
        # 生成该类别的二值图
        class_mask = (pred_resized == cls).astype(np.uint8)
        # 连通域标记
        labeled, num = label(class_mask, connectivity=2, return_num=True)
        for i in range(1, num+1):
            mask = (labeled == i).astype(np.uint8)
            area = np.sum(mask)
            if area < 100:   # 忽略太小的区域
                continue
            # 计算该区域的轮廓
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if len(contours) == 0:
                continue
            contour = contours[0]   # 取最大的轮廓
            # 计算该区域的HSV均值
            mean_hsv = cv2.mean(hsv, mask=mask)[:3]
            region = {
                'class_id': int(cls),
                'class_name': coco_classes[cls] if cls < len(coco_classes) else f"class_{cls}",
                'mask': mask.astype(bool),
                'contour': contour,
                'hsv_mean': mean_hsv,
                'area': area
            }
            ai_regions.append(region)

    print(f"AI识别到 {len(ai_regions)} 个区域")
    return ai_regions

# ---------- C: 弹窗确认 ----------
def user_confirmation(image, traditional_blocks, ai_regions, hue_threshold=10.0):
    """
    比较传统块与AI区域的重叠部分，对色调差异大于阈值的区域弹出确认窗口。
    用户可选择接受传统分类、接受AI分类或跳过。
    返回更新后的traditional_blocks列表（可能修改了某些块的color字段）。
    """
    if not ai_regions:
        print("未提供AI区域，跳过确认步骤。")
        return traditional_blocks

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    height, width = image.shape[:2]

    # 为每个传统块生成掩码（从轮廓）
    for block in traditional_blocks:
        mask = np.zeros((height, width), dtype=np.uint8)
        cv2.drawContours(mask, [block['contour']], -1, 255, -1)
        block['mask'] = mask.astype(bool)

    # 为每个AI区域生成掩码
    for region in ai_regions:
        if 'mask' not in region:
            mask = np.zeros((height, width), dtype=np.uint8)
            cv2.drawContours(mask, [region['contour']], -1, 255, -1)
            region['mask'] = mask.astype(bool)
        if 'hsv_mean' not in region:
            region_mask = region['mask']
            mean_hsv = cv2.mean(hsv, mask=region_mask.astype(np.uint8)*255)[:3]
            region['hsv_mean'] = mean_hsv

    # 查找重叠区域
    overlaps = []  # (trad_index, ai_index, overlap_mask)
    for ti, tblock in enumerate(traditional_blocks):
        tmask = tblock['mask']
        for ai, a_reg in enumerate(ai_regions):
            amask = a_reg['mask']
            intersection = np.logical_and(tmask, amask)
            inter_area = np.sum(intersection)
            if inter_area == 0:
                continue
            # 如果交集占各自面积的比例 > 20%，则认为对应
            t_area = tblock['area']
            a_area = np.sum(amask)
            if inter_area / t_area >= 0.2 or inter_area / a_area >= 0.2:
                overlaps.append((ti, ai, intersection))

    # 对有歧义的重叠进行确认
    modified_blocks = traditional_blocks.copy()
    for idx, (ti, ai, inter_mask) in enumerate(overlaps):
        # 计算重叠区域内的平均色调（传统块）
        inter_pixels_hue = hsv[..., 0][inter_mask]
        hue_trad = np.mean(inter_pixels_hue)
        # AI区域的全局平均色调
        hue_ai = ai_regions[ai]['hsv_mean'][0]
        diff = hue_difference(hue_trad, hue_ai)

        if diff > hue_threshold:
            # 弹出确认窗口
            print(f"\n重叠区域 {idx}: 传统色调={hue_trad:.1f}, AI色调={hue_ai:.1f}, 差异={diff:.1f}°")
            choice = confirm_single_region(image, inter_mask,
                                           traditional_blocks[ti], ai_regions[ai], idx)
            if choice is None:
                break  # 用户退出
            if choice == 'a':
                # 将传统块的颜色改为AI类别对应的颜色（这里简单映射为AI类别名）
                new_color = f"AI_{ai_regions[ai]['class_name']}"
                modified_blocks[ti]['color'] = new_color
                print(f"块 {ti} 颜色已更新为 {new_color}")
            elif choice == 't':
                print(f"保留块 {ti} 的传统分类 {traditional_blocks[ti]['color']}")
            # 若选择's'则跳过，不修改
        else:
            print(f"重叠区域 {idx} 色调差异 {diff:.1f}° 小于阈值，自动接受传统分类。")

    return modified_blocks

def confirm_single_region(image_bgr, overlap_mask, tblock, a_reg, index):
    """显示单个区域并等待用户按键"""
    img = image_bgr.copy()
    # 高亮重叠区域
    overlay = img.copy()
    overlay[overlap_mask] = (0, 0, 255)  # 红色
    alpha = 0.4
    img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
    # 绘制轮廓
    contours, _ = cv2.findContours(overlap_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(img, contours, -1, (0, 255, 0), 2)

    cv2.imshow(f"Confirmation {index}", img)
    print(f"区域 {index}: 传统分类='{tblock['color']}', AI分类='{a_reg.get('class_name', 'unknown')}'")
    print("按 't' 接受传统分类，按 'a' 接受AI分类，按 's' 跳过，按 ESC 退出")
    while True:
        key = cv2.waitKey(0) & 0xFF
        if key == ord('t'):
            cv2.destroyWindow(f"Confirmation {index}")
            return 't'
        elif key == ord('a'):
            cv2.destroyWindow(f"Confirmation {index}")
            return 'a'
        elif key == ord('s'):
            cv2.destroyWindow(f"Confirmation {index}")
            return 's'
        elif key == 27:  # ESC
            cv2.destroyAllWindows()
            return None

# ---------- D: 最终block分区完成（可添加后处理）----------
def finalize_blocks(blocks):
    """可选的最终处理，如再次合并微小区域等，目前直接返回"""
    return blocks

# ---------- E: 评分 ----------
def compute_score(blocks, image_shape):
    """根据最终块列表计算构图与色彩和谐度评分"""
    if not blocks:
        print("无有效块，无法评分。")
        return

    height, width = image_shape[:2]
    print("\n" + "=" * 60)
    print("Composition and Color Harmony Score")
    print("=" * 60)

    # 1. 锚点位置评分
    sorted_blocks = sorted(blocks, key=lambda b: (-b['saturation'][1], b['brightness'][1], -b['area']))
    anchor_blocks = sorted_blocks[:3]
    h_third, w_third = height / 3, width / 3
    cross_points = [(w_third, h_third), (2*w_third, h_third), (w_third, 2*h_third), (2*w_third, 2*h_third)]
    diag_length = np.sqrt(width**2 + height**2)
    threshold = diag_length * 0.1
    good_anchor_count = 0
    for block in anchor_blocks:
        x, y, w, h = block['position']
        center = (x + w/2, y + h/2)
        for cp in cross_points:
            dist = np.sqrt((center[0]-cp[0])**2 + (center[1]-cp[1])**2)
            if dist <= threshold:
                good_anchor_count += 1
                break
    position_score = (good_anchor_count / 3) * 100
    print(f"Anchor point position score: {position_score:.1f}/100 ({good_anchor_count}/3 near grid intersections)")

    # 2. 颜色面积比例评分
    color_area = defaultdict(float)
    for b in blocks:
        color_area[b['color']] += b['area']
    if len(color_area) >= 3:
        top_colors = sorted(color_area.items(), key=lambda x: -x[1])[:3]
        total_area = sum(area for _, area in top_colors)
        ratios = [area/total_area for _, area in top_colors]
        ideal = [0.6, 0.3, 0.1]
        error = sum(abs(r-i) for r,i in zip(ratios, ideal))
        ratio_score = max(0, 100 - error*100)
        print(f"Color area ratio score: {ratio_score:.1f}/100 (top colors: {[c for c,_ in top_colors]}, ratios: {[f'{r:.2f}' for r in ratios]})")
    else:
        ratio_score = 0
        print("Color area ratio score: 0/100 (less than 3 color categories)")

    # 3. 同类颜色一致性评分
    def hue_std(h_values):
        if len(h_values) < 2:
            return 0
        angles = np.deg2rad(np.array(h_values) * 2)
        complex_sum = np.sum(np.exp(1j * angles))
        avg_angle = np.angle(complex_sum)
        diffs = np.abs(np.angle(np.exp(1j * (angles - avg_angle))))
        std_rad = np.std(diffs)
        return np.rad2deg(std_rad)

    color_std_scores = []
    color_groups = defaultdict(list)
    for b in blocks:
        color_groups[b['color']].append(b['hue'])
    for color, hues in color_groups.items():
        std = hue_std(hues)
        score = 100 * (1 - std / 180)
        color_std_scores.append(score)
        print(f"  {color}: hue std = {std:.1f}°, consistency score = {score:.1f}")
    consistency_score = np.mean(color_std_scores) if color_std_scores else 0
    print(f"Overall color consistency score: {consistency_score:.1f}/100")

    total_score = 0.3*position_score + 0.4*ratio_score + 0.3*consistency_score
    print("-" * 60)
    print(f"COMPOSITE SCORE: {total_score:.1f}/100")
    print("=" * 60)

# ---------- 辅助函数：生成覆盖图并显示 ----------
def show_overlays(image, valid_blocks):
    """生成并显示HSV、Hue、Saturation、Value覆盖图"""
    if not valid_blocks:
        return
    hsv_img = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    res_hsv = image.copy()
    res_h = image.copy()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    gray_bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    res_v = gray_bgr.copy()
    res_s = image.copy()

    alphas = [1.0, 1.0, 0.1, 0.1]
    betas  = [0.5, 0.5, 0.9, 0.9]

    for block in valid_blocks:
        contour = block['contour']
        mean_hsv = block['mean_hsv']
        h, s, v = mean_hsv
        h_int, s_int, v_int = int(h), int(s), int(v)
        color_hsv_bgr = cv2.cvtColor(np.uint8([[[h_int, s_int, v_int]]]), cv2.COLOR_HSV2BGR)[0][0]
        color_h_bgr   = cv2.cvtColor(np.uint8([[[h_int, 255, 255]]]), cv2.COLOR_HSV2BGR)[0][0]
        color_s_bgr   = color_hsv_bgr
        color_v_bgr   = np.array([v_int, v_int, v_int], dtype=np.uint8)

        for img, color, alpha, beta in zip([res_hsv, res_h, res_s, res_v],
                                           [color_hsv_bgr, color_h_bgr, color_s_bgr, color_v_bgr],
                                           alphas, betas):
            layer = np.zeros_like(image)
            cv2.drawContours(layer, [contour], -1, color.tolist(), -1)
            cv2.addWeighted(img, alpha, layer, beta, 0, dst=img)

    cv2.imshow('HSV Overlay', res_hsv)
    cv2.imshow('Hue Overlay', res_h)
    cv2.imshow('Saturation Overlay', res_s)
    cv2.imshow('Value Overlay (on grayscale)', res_v)

# ---------- 主流程 ----------
def main_pipeline(image_path, use_ai=True, min_area=100, morph_kernel_size=5, merge_small=True, hue_threshold=10.0):
    # 读取图像
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Cannot read image {image_path}")
        return

    # ---- A: 传统识别 ----
    print("\n===== 步骤A: 传统计算机识别 =====")
    traditional_blocks, hsv, orig_img = traditional_recognition(image, min_area, morph_kernel_size, merge_small)

    # ---- B: AI识别 ----
    ai_regions = []
    if use_ai:
        print("\n===== 步骤B: AI识别 =====")
        ai_regions = ai_recognition(image)
        print(f"AI识别到 {len(ai_regions)} 个区域")
    else:
        print("\n===== 步骤B: 跳过AI识别 =====")

    # ---- C: 弹窗确认 ----
    if use_ai and ai_regions:
        print("\n===== 步骤C: 用户确认 =====")
        final_blocks = user_confirmation(image, traditional_blocks, ai_regions, hue_threshold)
    else:
        final_blocks = traditional_blocks

    # ---- D: 最终block分区完成 ----
    print("\n===== 步骤D: 最终分区完成 =====")
    final_blocks = finalize_blocks(final_blocks)

    # ---- E: 评分 ----
    print("\n===== 步骤E: 评分 =====")
    compute_score(final_blocks, image.shape)

    # 生成覆盖图并显示结果
    valid_blocks = [b for b in final_blocks if 'contour' in b]
    show_overlays(image, valid_blocks)

    # 绘制最终轮廓结果
    result_image = image.copy()
    for block in final_blocks:
        color_bgr = get_color_bgr(block['color'])
        cv2.drawContours(result_image, [block['contour']], -1, color_bgr, 2)
        x, y, w, h = block['position']
        cv2.rectangle(result_image, (x, y), (x + w, y + h), color_bgr, 1)
        # 可添加文字标注
    cv2.imshow('Final Detection', result_image)
    cv2.imwrite('final_detection.jpg', result_image)

    print("\n按任意键关闭所有窗口...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    script_dir = os.getcwd()
    image_path = os.path.join(script_dir, 'YeLu', '1.jpg')
    print(f"Current directory: {script_dir}")
    print(f"Image path: {image_path}")

    # 启用AI识别，可调整色调差异阈值
    main_pipeline(image_path, use_ai=True, hue_threshold=10.0)

Current directory: /Users/DELL/Downloads/pro 2/AI/Gou Tu
Image path: /Users/DELL/Downloads/pro 2/AI/Gou Tu/YeLu/1.jpg

===== 步骤A: 传统计算机识别 =====
Merging 343 small blocks...
Combining 343 remaining small blocks into one block using convex hull...
Generated 1 merged blocks (including remaining).

===== 传统识别完成 =====
有效色块数量: 16

===== 步骤B: AI识别 =====


Using cache found in /Users/DELL/.cache/torch/hub/pytorch_vision_main
/Applications/anicoda/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Applications/anicoda/anaconda3/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


AI识别到 1 个区域
AI识别到 1 个区域

===== 步骤C: 用户确认 =====

重叠区域 0: 传统色调=10.5, AI色调=24.5, 差异=14.1°
区域 0: 传统分类='Orange/Yellow', AI分类='bird'
按 't' 接受传统分类，按 'a' 接受AI分类，按 's' 跳过，按 ESC 退出


2026-03-17 19:50:09.984 python[1168:11338] Warning: Window move completed without beginning
